In [15]:
!pip install groq pymupdf jsonschema

In [16]:
import json
from groq import Groq
from google.colab import userdata
from jsonschema import validate, ValidationError

# Initialize client
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Define the JSON Schema for enforcement
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "citations": {
            "type": "array",
            "items": {"type": "string"}
        },
        "confidence": {"type": "number", "minimum": 0, "maximum": 1}
    },
    "required": ["answer", "citations", "confidence"]
}

In [17]:
def find_pages_by_keyword(pdf_path, keyword, window=2):
    doc = fitz.open(pdf_path)
    found_pages = []
    for i, page in enumerate(doc):
        if keyword.lower() in page.get_text().lower():
            found_pages.append(i)

    if not found_pages:
        return "Keyword not found."

    # Group nearby pages to provide context
    ranges = []
    for p in found_pages:
        start = max(0, p - window)
        end = min(len(doc) - 1, p + window)
        ranges.append((start, end))

    return ranges

def extract_specific_pages(pdf_path, page_ranges):
    doc = fitz.open(pdf_path)
    text = ""
    processed = set()
    for start, end in page_ranges:
        for p_num in range(start, end + 1):
            if p_num not in processed:
                text += f"--- Page {p_num + 1} ---\n" + doc[p_num].get_text()
                processed.add(p_num)
    return text

# Identify pages for the Battle of Badr
target_keyword = "Badr"
page_ranges = find_pages_by_keyword('/content/Seerat e Mustafa_new.pdf', target_keyword)
print(f"Found '{target_keyword}' in ranges: {page_ranges}")

if isinstance(page_ranges, list):
    # Extract text from those specific sections
    pdf_context = extract_specific_pages('/content/Seerat e Mustafa_new.pdf', page_ranges[:3]) # Limit to first 3 matches for efficiency

Found 'Badr' in ranges: [(7, 11), (8, 12), (10, 14), (102, 106), (105, 109), (107, 111), (108, 112), (110, 114), (111, 115), (112, 116), (113, 117), (118, 122), (132, 136), (147, 151), (149, 153), (168, 172), (215, 219), (245, 249), (250, 254), (251, 255), (252, 256), (253, 257), (258, 262), (259, 263), (261, 265), (262, 266), (267, 271), (268, 272), (270, 274), (274, 278), (275, 279), (276, 280), (279, 283), (280, 284), (281, 285), (282, 286), (283, 287), (284, 288), (286, 290), (288, 292), (289, 293), (292, 296), (293, 297), (295, 299), (296, 300), (297, 301), (298, 302), (299, 303), (300, 304), (301, 305), (302, 306), (303, 307), (304, 308), (305, 309), (307, 311), (308, 312), (309, 313), (311, 315), (315, 319), (318, 322), (320, 324), (330, 334), (341, 345), (343, 347), (344, 348), (345, 349), (350, 354), (351, 355), (363, 367), (375, 379), (376, 380), (383, 387), (398, 402), (414, 418), (475, 479), (482, 486), (501, 505), (517, 521), (527, 531), (528, 532), (543, 547), (636, 640),

In [18]:
def get_enforced_answer(query, context):
    system_prompt = (
        "You are a helpful assistant. Use the provided context to answer the user question. "
        "Your response MUST be in valid JSON format matching this schema: "
        "{\"answer\": \"string\", \"citations\": [\"string\"], \"confidence\": number}. "
        "Only include information found in the context. If you use a specific fact, "
        "include the exact sentence or page reference in the citations array."
    )

    user_prompt = f"Context:\n{context}\n\nQuestion: {query}"

    # Updated model to a currently supported version
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        model="llama-3.3-70b-versatile",
        response_format={"type": "json_object"}
    )

    raw_response = chat_completion.choices[0].message.content
    data = json.loads(raw_response)

    # Validate against schema
    try:
        validate(instance=data, schema=ANSWER_SCHEMA)
        return data
    except ValidationError as e:
        return {"error": "Schema validation failed", "details": str(e)}

# Example usage
result = get_enforced_answer("What are the main events of the Battle of Badr?", pdf_context)
display(result)

{'answer': 'The main events of the Battle of Badr include the preamble to the battle, departure, mashwarah with the Sahaabah, selfless sermon of Miqdaad bin Aswad, valiant speech of S‘ad bin Mu’aaz, the battle itself, slaying of ‘Utbah, Shaybah and Waleed, Rasulullah’s Dua for Victory, descent of the angels to assist the Muslims, slaying of Abu Jahal, searching for Abu Jahal’s body after the victory, the prisoners of Badr, disposing of the corpses in the well of Badr, despatching a messenger to Madinah with news of victory, distribution of the booty, and distribution of the war captives amongst the Muslims.',
 'citations': ['Chapter 12: Battle of Badr – 2 A.H.',
  'Preamble to the Battle of Badr (Page 223)',
  'Departure (Page 224)',
  'Mashwarah with the Sahaabah \uf04d and their Staunchly Devoted Discourses (Page 227)',
  'Selfless sermon of Miqdaad bin Aswad (Page 227)',
  'The Valiant Speech of S‘ad bin Mu’aaz (Page 228)'],
 'confidence': 0.95}